# 2.5 Your turn: your own chat

Everything through [02.4](02.4-same_trap.ipynb) ran on showcase data, chosen so the pattern
was certain to be there. This notebook has one worked example, to prove the pipes are
connected, and then it is your data and your question.

Your own chat has far fewer people than the IRC corpus does, which makes 02.4's trap sharper
rather than softer — a channel of five people has no room for "the eight regulars" to hide
in, and every person you drop from a comparison moves the answer more, not less.

In [ ]:
import pandas as pd
from goad_toolkit.visualizer import PlotSettings
from notebooktester import param

from scripts.plots import BarPlot
from wa_analyzer.data import load_own_chat

## 2.5.1 Loading your data

`load_own_chat()` returns `None` rather than raising when there is no export yet — see
[01.3](../lesson1/01.3-your-own-chat.ipynb). Guarding every cell below with
`if own is not None:` would work, but it would also mean rewriting that guard into every
single cell you add from here on, and getting exactly one of them wrong is how a "your turn"
notebook silently stops testing anything.

Same fix as 01.3 used: decide once, right here, and give every cell after this one real data
to work with either way.

In [ ]:
own = load_own_chat()

REQUIRE_OWN_CHAT = param(True, test=False)
if own is None and REQUIRE_OWN_CHAT:
    raise RuntimeError(
        "No chat of your own yet. Run notebook 01.3-your-own-chat.ipynb first, then set "
        "`current` in config.toml to the file it writes."
    )
elif own is None:
    # No config.toml and no export -- the path notebooktester's CI run takes, since it
    # checks out a clean repo. A tiny stand-in keeps every cell below written as if `own`
    # is always real data, instead of guarding each one separately.
    own = pd.DataFrame({
        "timestamp": pd.to_datetime([
            "2024-01-01 09:00", "2024-01-01 09:05", "2024-01-02 20:00", "2024-01-02 20:04",
        ]),
        "author": ["Alex", "Sam", "Alex", "Sam"],
        "message": [
            "morning!", "hey, see the link https://example.com?",
            "yes indeed", "same time tomorrow?",
        ],
    })

own["length"] = own.message.str.len()
own.head()

## 2.5.2 What's already loaded

`BarPlot` — the class 02.2 derived — is imported above from `scripts/plots.py`, ready to use.
`BarPlotWithError`, 02.4's error-bar version, lives in the same file if the comparison you
pick needs one (`from scripts.plots import BarPlotWithError`). One worked example first, so
you know the pipes connect before you start changing things:

In [ ]:
counts = own.author.value_counts().rename_axis("author").reset_index(name="n")

base_settings = PlotSettings(
    figsize=(8, 4),
    title="Messages per person",
    xlabel="messages",
    ylabel="",
)
BarPlot(base_settings).plot(data=counts, x="n", y="author", color="#cccccc")

## 2.5.3 Brainstorming with `goad`

This repo has the `goad` MCP server connected — the same one behind
[CLAUDE.md](../../CLAUDE.md)'s coaching policy. Three tools worth knowing about here, all on
the assistant side of this notebook rather than in it:

- **`goad_analysis_checklist`** walks the six-stage method — the question, the data, the
  shape, the encoding, the critique, the verification — one stage at a time, and will not
  move to the next until you've answered the current one in your own words. Ask your
  assistant to run it for a comparison you want to make in your own chat, and expect to be
  interviewed rather than handed a finding.
- **`goad_search`** / **`goad_list_concepts`** / **`goad_get_concept`** look up a concept
  directly — "which chart family fits a question about who talks to whom", say, or the
  Simpson's-paradox chapter again if 02.3 is worth a second read against your own data.
- **`goad_critique_visual`** runs the four-part visual-critique checklist once you have a
  chart to defend — the same "grey first, is the finding written on the plot, what could you
  delete" questions 02.2 and 02.3 used, put to *your* chart instead of the showcase ones.

> **Your turn.**
>
> 1. Make one bar chart that answers a question you actually have about your chat. Order it,
>    grey it, and colour the one bar your title is about — 02.2's guidelines, applied to data
>    nobody chose for you.
> 2. Pick a comparison between two people or two groups. Compute it counting messages, then
>    counting people, the way 02.4 did. Do they agree?
> 3. Before you trust either chart: how many *people* does the claim actually rest on? If the
>    honest answer is two or three, that is worth knowing before the chart is.

In [ ]:
# >>> Your turn: swap the comparison below for one you actually want to know. <<<
# The line below runs as-is so the notebook has something to show either way — edit x, y,
# and the data it's computed from.

comparison = own.groupby("author").length.mean().rename("mean_length").reset_index()

your_settings = PlotSettings(
    figsize=(8, 4),
    title="Mean message length by author",
    xlabel="mean length (characters)",
    ylabel="",
)
BarPlot(your_settings).plot(data=comparison, x="mean_length", y="author", color="#cccccc")

## 2.5.4 Reflect

- Did counting people change the answer from counting messages? If it did, were you
  surprised, or had you already half-expected it?
- Is there a variable — active period, group role, which sub-conversation someone is in —
  that might split your comparison the way department split Berkeley's admissions in 02.3?
  You don't have to check it now. Just: does one come to mind?
- Look at the chart you made in 2.5.3. If a stranger saw only the chart, thirty seconds, no
  narration from you — would they land on the finding you actually have? If you're not sure,
  `goad_critique_visual` exists for exactly this question, and it's a fair thing to ask your
  assistant to run right now.
- What would it take to change your mind about the comparison you just made? If nothing
  would, it might not be a finding yet.

---

**Where this goes next.** Lesson 3 takes the same comparisons and puts time on the x-axis,
where a new failure appears: the difference between a pattern and the way you smoothed it.